# Task 1.2 — Key Assumptions of DP-means
**Paper:** Revisiting k-means: New Algorithms via Bayesian Nonparametrics (Kulis & Jordan, ICML 2012)

## Assumption 1

**Assumption:** Each cluster is isotropic (spherical) — it has equal variance in all directions. Formally, all mixture components share the same scalar covariance σ²I.

**Why the method needs it:** DP-means uses squared Euclidean distance to assign points and update centers (Algorithm 1, line 4; Eq. 9). This distance metric only produces Voronoi partitions that are optimal for spherical Gaussians. If clusters have different shapes or orientations, Euclidean distance mis-measures which cluster a point truly belongs to, and the assignment step loses its probabilistic justification as a MAP estimate under the DP-GMM.

**Violation scenario:** A dataset containing elongated, banana-shaped, or crescent-shaped clusters (such as the two-moons dataset). DP-means will incorrectly fragment elongated clusters into multiple small spherical pieces or incorrectly merge nearby crescents into a single cluster, regardless of how λ is tuned.

**Paper reference:** Section 4, Equation 9 — the DP-means objective is a sum of squared Euclidean distances, which directly implies equal isotropic covariance. The derivation from DP-GMM in Theorem 1 explicitly assumes each Gaussian component has covariance σ²I.

---

## Assumption 2

**Assumption:** A single global scalar λ governs new-cluster creation for the entire dataset uniformly.

**Why the method needs it:** The algorithm compares every point's distance against the same threshold λ at every iteration (Algorithm 1, line 5). This assumes all clusters have similar density, scale, and spread — because a λ appropriate for one cluster will be either too tight or too loose for a cluster with different variance.

**Violation scenario:** A dataset containing clusters of very different scales — for example, one tightly packed micro-cluster of 10 points and one diffuse macro-cluster of 500 points spread over a large area. A λ small enough to capture the micro-cluster will fragment the macro-cluster into dozens of spurious sub-clusters; a λ large enough for the macro-cluster will cause the micro-cluster to be absorbed into a neighbor.

**Paper reference:** Section 3, the definition of the single λ parameter and its role as a uniform penalty. No mechanism for per-cluster λ adaptation is introduced in the paper.

---

## Assumption 3

**Assumption:** The clusters are well-separated relative to their within-cluster variance — i.e., the small-variance asymptotic (σ²→0) derivation is approximately valid for the data at hand.

**Why the method needs it:** The entire DP-means algorithm is derived by taking σ²→0 in the Gibbs sampler for a DP-GMM (Section 4, Theorem 1). In this limit, soft posterior membership probabilities collapse to hard 0/1 indicator assignments. If clusters overlap significantly (σ² is not small relative to inter-cluster distance), the hard assignment step introduces large approximation error relative to the DP-GMM, and the algorithm loses its Bayesian justification.

**Violation scenario:** A dataset with heavy cluster overlap, where points near cluster boundaries have genuinely ambiguous membership (e.g., two Gaussians with the same mean but slightly different covariances, or two clusters separated by less than one standard deviation). DP-means will produce unstable, arbitrary assignments at boundaries and may not converge to a meaningful partition.

**Paper reference:** Section 4, Theorem 1 and the surrounding derivation. The paper explicitly states the asymptotics are taken as σ²→0, and the resulting algorithm is only guaranteed to minimize the DP-means objective — not to recover the true cluster structure when the σ²→0 approximation is poor.
